####Daily Sales Performance

In [0]:
%sql
--select * from dd_hr_sandpit.ecommerce_project.silver_clickstream where event_type = 'purchase'
select * from dd_hr_sandpit.ecommerce_project.gold_funnel_conversion 

In [0]:
%sql
-- Create or replace the Gold Daily Revenue table
CREATE OR REPLACE TABLE dd_hr_sandpit.ecommerce_project.gold_daily_revenue 
COMMENT "Gold layer: Materialised daily revenue aggregated by product category and name"
AS
SELECT 
    DATE(c.timestamp) AS event_date,
    p.category AS product_category,
    p.product_name,
    SUM(p.price) AS total_revenue,
    COUNT(c.event_id) AS total_purchase_count
FROM 
    dd_hr_sandpit.ecommerce_project.silver_clickstream c
INNER JOIN 
    dd_hr_sandpit.ecommerce_project.silver_products p 
    ON c.product_id = p.product_id
WHERE 
    c.event_type = 'purchase'
GROUP BY 
    ALL;


####User Funnel Conversion

In [0]:
%sql
-- Create or replace the Gold Funnel Conversion table
CREATE OR REPLACE TABLE dd_hr_sandpit.ecommerce_project.gold_funnel_conversion
COMMENT "Gold layer: User funnel analysis showing drop-off rates from views to purchases"
AS
WITH user_event_counts AS (
    SELECT 
        customer_id,
        COUNT(CASE WHEN event_type = 'view' THEN 1 END) AS total_views,
        COUNT(CASE WHEN event_type = 'add_to_cart' THEN 1 END) AS total_adds,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS total_purchases
    FROM 
        dd_hr_sandpit.ecommerce_project.silver_clickstream
    GROUP BY 
        customer_id
)
SELECT 
    COUNT(DISTINCT customer_id) AS total_unique_users,
    SUM(total_views) AS aggregate_views,
    SUM(total_adds) AS aggregate_adds,
    SUM(total_purchases) AS aggregate_purchases,
    
    -- Conversion rates (handling division by zero safely)
    ROUND(IFNULL(SUM(total_adds) / NULLIF(SUM(total_views), 0), 0) * 100, 2) AS view_to_cart_percentage,
    ROUND(IFNULL(SUM(total_purchases) / NULLIF(SUM(total_adds), 0), 0) * 100, 2) AS cart_to_purchase_percentage,
    ROUND(IFNULL(SUM(total_purchases) / NULLIF(SUM(total_views), 0), 0) * 100, 2) AS overall_conversion_percentage
FROM 
    user_event_counts;


In [0]:
%sql
SELECT 'gold_daily_revenue' AS table_name, COUNT(*) AS row_count FROM dd_hr_sandpit.ecommerce_project.gold_daily_revenue
UNION ALL
SELECT 'gold_funnel_conversion' AS table_name, COUNT(*) AS row_count FROM dd_hr_sandpit.ecommerce_project.gold_funnel_conversion;

